# Drosophila Brain Simulation - GPU Accelerated

138,639 LIF neurons from FlyWire v783 on free Colab T4 GPU.

Expected: ~90x faster than CPU (1.2ms/step vs 90ms/step)

## 1. Setup

In [0]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
os.chdir("/content")
!rm -rf /content/flywire-neuro
!git lfs install --skip-smudge
!git clone https://github.com/pusulamkendim/flywire-neuro.git /content/flywire-neuro
os.chdir("/content/flywire-neuro")
!pip install -q fastapi uvicorn websockets pydantic pandas pyarrow nest_asyncio
print("CWD:", os.getcwd())

## 2. Download data from Zenodo

In [ ]:
import os, urllib.request

# Data files are in the repo (pushed to git)
DATA_DIR = "fly-brain-embodied/data"
COMP_FILE = f"{DATA_DIR}/2025_Completeness_783.csv"
CONN_FILE = f"{DATA_DIR}/2025_Connectivity_783.parquet"
ANN_FILE = "data/neuron_annotations.tsv"

# Verify repo data exists
assert os.path.exists(COMP_FILE), f"Missing {COMP_FILE} — re-clone the repo"
assert os.path.exists(CONN_FILE), f"Missing {CONN_FILE} — re-clone the repo"

# Only annotation TSV needs downloading (too large for git, from FlyWire GitHub)
os.makedirs("data", exist_ok=True)
if not os.path.exists(ANN_FILE):
    print("Downloading neuron_annotations.tsv...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/flyconnectome/flywire_annotations/main/supplemental_files/Supplemental_file1_neuron_annotations.tsv",
        ANN_FILE)
    print(f"  Done ({os.path.getsize(ANN_FILE)//1024//1024}MB)")
else:
    print(f"OK {ANN_FILE}")

print(f"OK {COMP_FILE} ({os.path.getsize(COMP_FILE)//1024//1024}MB)")
print(f"OK {CONN_FILE} ({os.path.getsize(CONN_FILE)//1024//1024}MB)")
print("All data ready!")

## 3. Load brain on GPU

In [ ]:
import sys, time, importlib.util
import torch, numpy as np, pandas as pd

def force_import(name, path):
    """Load module from file path (bypasses Colab sys.path issues)."""
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

CODE_DIR = "fly-brain-embodied/code"
BRAIN_DIR = "fly-brain-embodied"

benchmark = force_import("benchmark", f"{CODE_DIR}/benchmark.py")
benchmark.COMP_PATH = COMP_FILE
benchmark.CONN_PATH = CONN_FILE
benchmark.DATA_DIR = DATA_DIR

run_pytorch = force_import("run_pytorch", f"{CODE_DIR}/run_pytorch.py")
brain_body_bridge = force_import("brain_body_bridge", f"{BRAIN_DIR}/brain_body_bridge.py")

TorchModel = run_pytorch.TorchModel
MODEL_PARAMS = run_pytorch.MODEL_PARAMS
DT = run_pytorch.DT
get_weights = run_pytorch.get_weights
get_hash_tables = run_pytorch.get_hash_tables
DN_NEURONS = brain_body_bridge.DN_NEURONS
DN_GROUPS = brain_body_bridge.DN_GROUPS
STIMULI = brain_body_bridge.STIMULI
DNRateDecoder = brain_body_bridge.DNRateDecoder

flyid2i, i2flyid = get_hash_tables(COMP_FILE)
num_neurons = len(flyid2i)
device = "cuda" if torch.cuda.is_available() else "cpu"
t0 = time.time()
weights = get_weights(CONN_FILE, COMP_FILE, DATA_DIR).to(device=device)
print(f"Neurons: {num_neurons:,} | Device: {device} | Loaded in {time.time()-t0:.1f}s")

## 4. Benchmark

In [0]:
model = TorchModel(batch=1, size=num_neurons, dt=DT, params=MODEL_PARAMS, weights=weights, device=device)
state = model.state_init()
rates = torch.zeros(1, num_neurons, device=device)
for _ in range(5):
    with torch.no_grad(): state = model(rates, *state)
if device=="cuda": torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    with torch.no_grad(): state = model(rates, *state)
if device=="cuda": torch.cuda.synchronize()
ms = (time.time()-t0)/100*1000
print(f"Speed: {ms:.2f} ms/step | {1.0/(ms*10/1000):.0f} fps | {90/ms:.0f}x vs CPU")

## 5. Setup populations + server

In [0]:
ann = pd.read_csv("data/neuron_annotations.tsv", sep="\t", low_memory=False)
def ids_to_tensor(root_ids):
    return torch.tensor([flyid2i[nid] for nid in root_ids if nid in flyid2i], dtype=torch.long, device=device)
mbon = ann[ann["cell_class"] == "MBON"]
ser = ann[(ann["top_nt"] == "serotonin") & (~ann["cell_class"].isin(["olfactory","visual","mechanosensory","unknown_sensory","gustatory"]))]
octo = ann[(ann["top_nt"] == "octopamine") & (~ann["cell_class"].isin(["olfactory","visual","mechanosensory","unknown_sensory","gustatory"]))]
pop_tensors = {
    "pam": ids_to_tensor(ann[ann["cell_type"].str.startswith("PAM", na=False)]["root_id"]),
    "ppl1": ids_to_tensor(ann[ann["cell_type"].str.startswith("PPL1", na=False)]["root_id"]),
    "mbon_approach": ids_to_tensor(mbon[mbon["top_nt"] == "acetylcholine"]["root_id"]),
    "mbon_avoidance": ids_to_tensor(mbon[mbon["top_nt"] == "glutamate"]["root_id"]),
    "mbon_suppress": ids_to_tensor(mbon[mbon["top_nt"] == "gaba"]["root_id"]),
    "serotonin": ids_to_tensor(ser["root_id"]),
    "octopamine": ids_to_tensor(octo["root_id"]),
    "gaba": ids_to_tensor(ann[ann["top_nt"] == "gaba"]["root_id"]),
    "ach": ids_to_tensor(ann[ann["top_nt"] == "acetylcholine"]["root_id"]),
    "glut": ids_to_tensor(ann[ann["top_nt"] == "glutamate"]["root_id"]),
}
stim_map = {sn: [flyid2i[nid] for nid in si["neurons"] if nid in flyid2i] for sn, si in STIMULI.items()}
dn_map = {name: flyid2i[fid] for name, fid in DN_NEURONS.items() if fid in flyid2i}
print(f"Populations: {len(pop_tensors)} | DN: {len(dn_map)} | Stimuli: {list(stim_map.keys())}")

In [ ]:
import asyncio, json, threading
from pathlib import Path
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI()

# Serve full frontend (3D, brain vis, dashboard, CSS, JS, GLB models)
FRONTEND_DIR = Path("web/frontend")
app.mount("/static", StaticFiles(directory=str(FRONTEND_DIR)), name="static")

# --- Shared state ---
active_stimuli = set()
brain_lock = threading.Lock()
brain_running = False
brain_thread = None
frame_queue = asyncio.Queue(maxsize=200)
_loop = None

# --- API endpoints (matching frontend expectations) ---

@app.get("/")
async def index():
    return FileResponse(str(FRONTEND_DIR / "index.html"))

@app.get("/api/scenarios")
async def get_scenarios():
    return []  # brain mode only — no scenario replay on Colab

@app.post("/api/brain")
async def start_brain(body: dict = None):
    global brain_running, brain_thread, _loop, active_stimuli
    # Stop existing brain
    brain_running = False
    if brain_thread and brain_thread.is_alive():
        brain_thread.join(timeout=2)
    # Drain queue
    while not frame_queue.empty():
        try: frame_queue.get_nowait()
        except: break

    _loop = asyncio.get_running_loop()
    initial = body.get('stimuli', []) if body else []
    with brain_lock:
        active_stimuli = set(initial)

    brain_running = True
    brain_thread = threading.Thread(target=_brain_loop, daemon=True)
    brain_thread.start()
    return {"status": "brain_started", "initial_stimuli": initial}

@app.post("/api/stop")
async def stop_sim():
    global brain_running
    brain_running = False
    return {"status": "stopped"}

@app.get("/api/walk_cache")
async def get_walk_cache():
    """Return cached walk frames for brain-driven 3D animation."""
    cache_path = Path("web/backend/walk_cache/walk_5.0s.json")
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)
    return {"geom_names": [], "frames": []}

# --- Brain simulation (GPU background thread) ---

def _brain_loop():
    """Run 138,639 LIF neurons on GPU, push frames to async queue."""
    global brain_running
    decoder = DNRateDecoder(window_ms=50.0, dt_ms=DT, max_rate=200.0)
    r = torch.zeros(1, num_neurons, device=device)
    st = model.state_init()
    sa = torch.zeros(num_neurons, device=device)
    prev = set()
    step = 0

    while brain_running:
        with brain_lock:
            cur = active_stimuli.copy()
        if cur != prev:
            r.zero_()
            for sn in cur:
                if sn in stim_map:
                    for idx in stim_map[sn]:
                        r[0, idx] = STIMULI[sn]["rate"]
            prev = cur.copy()

        with torch.no_grad():
            st = model(r, *st)
        sa += st[2][0]
        decoder.update({n: st[2][0, i].item() for n, i in dn_map.items()})
        step += 1

        if step % 10 == 0:
            dn = {g: round(decoder.get_group_rate(g), 4) for g in DN_GROUPS}
            gf = dn.get("escape", 0)
            beh = ("escape" if gf > 0.06
                   else "grooming" if dn.get("groom", 0) > 0.02
                   else "feeding" if dn.get("feed", 0) > 0.05
                   else "walking" if dn.get("forward", 0) > 0.01
                   else "idle")
            pop = {n: int(sa[i].sum().item()) for n, i in pop_tensors.items()}
            tot = int(sa.sum().item())
            sa.zero_()

            frame = {
                "event": "brain_frame",
                "t_ms": round(step * DT, 1),
                "brain_steps": step,
                "dn": dn,
                "pop": pop,
                "total_spikes": tot,
                "behavior_mode": beh,
                "active_stimuli": list(cur),
            }
            try:
                asyncio.run_coroutine_threadsafe(
                    frame_queue.put(frame), _loop
                ).result(timeout=1)
            except:
                pass

# --- WebSocket (matches frontend's /ws/sim) ---

@app.websocket("/ws/sim")
async def ws_sim(websocket: WebSocket):
    await websocket.accept()

    async def receive_commands():
        global active_stimuli
        try:
            while True:
                msg = await websocket.receive_text()
                cmd = json.loads(msg)
                if cmd.get('cmd') == 'set_stimuli':
                    with brain_lock:
                        active_stimuli = set(cmd.get('stimuli', []))
        except WebSocketDisconnect:
            pass
        except:
            pass

    async def send_frames():
        try:
            while True:
                data = await frame_queue.get()
                await websocket.send_json(data)
        except:
            pass

    await asyncio.gather(receive_commands(), send_frames())

print("Server ready — full 3D frontend with brain visualization")

## 6. Start with public URL

In [0]:
# cloudflared - no account needed
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess, time as _time
proc = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
_time.sleep(8)
for line in proc.stderr:
    l = line.decode()
    if "trycloudflare.com" in l:
        url = [w for w in l.split() if "trycloudflare.com" in w][0]
        print(f"\nPublic URL: {url}")
        print("Open this in your browser!")
        break

In [ ]:
# Start server (non-blocking for Colab)
import nest_asyncio
nest_asyncio.apply()
print("Starting brain server on GPU...")
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()